# 1.3 毕昇编译器基础

## 本节学习目标

- 区分 clang/clang++ 与 bisheng 的使用场景
- 识别 Ascend C 的架构参数和编译产物

## 必要背景

鲲鹏毕昇 Host 编译器安装包提供 `clang`/`clang++`，因此官方使用 `clang -v` 验证安装；CANN 异构毕昇编译器使用 `bisheng` 编译 `.asc` 源文件。两者属于不同工具链，不能互相替代。Ascend C 的目标架构参数必须依据实际产品和当前 CANN 版本选择。

## 安装鲲鹏毕昇 Host 编译器

CANNLab A3 镜像可能只预装 CANN `bisheng`。若 `command -v clang` 没有输出，请先按[鲲鹏毕昇编译器官方安装指南](https://www.hikunpeng.com/document/detail/zh/kunpengdevps/compilation/ug-bisheng/kunpengbisheng_06_0005.html)下载 ARM64 软件包并校验完整性，再解压到可写目录。以下变量中的目录名应替换为实际解压目录：

```bash
export BISHENG_HOST_HOME="$HOME/opt/BiShengCompiler-<版本>-aarch64-linux"
export PATH="$BISHENG_HOST_HOME/bin:$PATH"
export LD_LIBRARY_PATH="$BISHENG_HOST_HOME/lib:$BISHENG_HOST_HOME/lib/aarch64-unknown-linux-gnu:${LD_LIBRARY_PATH:-}"
hash -r
clang -v
clang++ -v
```

`clang -v` 和 `clang++ -v` 的输出应包含 BiSheng 编译器标识。

## 关键区别

毕昇可同时处理 Host/Device 异构源，但本节只建立最小流程；后续真实算子仍应使用其工程自带 CMake/build 规则。

## 查看版本和帮助

从当前章节目录执行下面的 Cell，并对照随后给出的检查点阅读输出。

In [ ]:
import shutil
import subprocess

if shutil.which("npu-smi"):
    subprocess.run(["npu-smi", "info"], check=False)

for tool in ("clang", "clang++", "bisheng"):
    path = shutil.which(tool)
    print(f"{tool}: {path or 'not found'}")
    if path:
        subprocess.run([path, "--version"], check=False)

if not shutil.which("clang++"):
    print("未找到鲲鹏毕昇 Host 编译器，请先按上面的官方指南安装。")


## 查看最小源文件

从当前章节目录执行下面的 Cell，并对照随后给出的检查点阅读输出。

In [ ]:
!sed -n '1,120p' src/hello_world.asc
# 在目标环境确认架构参数后执行，例如：
# bisheng src/hello_world.asc --npu-arch=<实际架构> -o build/hello_world

## 预期现象与结果分析

`clang -v` 和 `clang++ -v` 用于确认鲲鹏毕昇 Host 编译器；`bisheng --version` 用于确认 CANN 异构编译器。若前两项缺失而 `bisheng` 存在，说明当前环境只有 CANN 编译链，仍需完成 Host 编译器安装。

## 课后实践

根据目标环境的 `bisheng --help` 找到架构参数，写出编译命令并说明每个选项。

参考答案见 `answer/01.03_answer.md`。

In [ ]:
from pathlib import Path
print(Path('answer/01.03_answer.md').read_text(encoding='utf-8'))

## 实验步骤：把本节落实到 `src/`

1. 从 Notebook 当前目录确认 `src/` 存在。
2. 打开本节 Code Cell 定位的片段，在完整文件中找到所属函数和调用者。
3. 对照工程职责：`hello_host.cpp` 是 Host C++ 最小程序；`hello_world.asc` 用于识别毕昇/Ascend C 编译边界。
4. 执行到本节对应阶段：检查环境与工具版本 → 阅读两个源码 → 编译运行 Host 程序 → 记录工具链 → 判断哪些步骤需要 CANN。
5. 每次只改变一个变量，固定输入、构建类型、warmup/repeat。
6. 记录：工具、版本/路径、退出码、Host 输出、Ascend C 编译条件。
7. 先检查退出码和正确性，再比较时间；历史结果不是本机输出。

### 本节完成标准

能够用真实文件和函数解释机制，给出可复现命令、至少一组结果或环境受限诊断，并说明结果如何进入下一节。
